# Forward Feature Selection (FFS) with Config Presets
This notebook provides:
- Forward Feature Selection with CV
- Pluggable preprocessing: imputation + encoding (OneHot, Target, native CatBoost/LightGBM/XGB)
- Pluggable models: LogisticRegression, XGBoost, LightGBM, CatBoost, Bayesian classifiers
- Config **presets** so you can choose options instead of hand-writing full configs


## Imports

In [61]:
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from random import shuffle
from typing import Any, Dict, List, Optional, Tuple, Literal

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from category_encoders import TargetEncoder
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, KFold, cross_val_score, cross_validate
from sklearn.naive_bayes import GaussianNB, BernoulliNB, ComplementNB, CategoricalNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_curve,
    roc_curve
)
from sklearn.ensemble import VotingClassifier, StackingClassifier
from tqdm import tqdm
from xgboost import XGBClassifier, XGBRegressor


## Native preprocessors/wrappers (CatBoost / LightGBM / XGB)

In [62]:
class NativeDataFramePreprocessor(BaseEstimator, TransformerMixin):
    """
    Keeps data as pandas.DataFrame with:
    - numeric imputed, coerced numeric
    - categorical imputed, cast to 'category'
    Useful for models that can consume DataFrames and accept categorical metadata.
    """

    def __init__(self, numeric_cols: Tuple[str, ...], categorical_cols: Tuple[str, ...],
                 num_imputer: BaseEstimator, cat_imputer: Optional[BaseEstimator]):
        self.numeric_cols = tuple(numeric_cols)
        self.categorical_cols = tuple(categorical_cols)
        self.num_imputer = num_imputer
        self.cat_imputer = cat_imputer

    def fit(self, X: pd.DataFrame, y=None):
        X = X.copy()
        if self.numeric_cols:
            self.num_imputer.fit(X[list(self.numeric_cols)])
        if self.categorical_cols and self.cat_imputer is not None:
            self.cat_imputer.fit(X[list(self.categorical_cols)])
        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()
        parts = []

        if self.numeric_cols:
            num_arr = self.num_imputer.transform(X[list(self.numeric_cols)])
            num_df = pd.DataFrame(num_arr, columns=list(self.numeric_cols), index=X.index)
            for c in self.numeric_cols:
                num_df[c] = pd.to_numeric(num_df[c], errors="coerce")
            parts.append(num_df)

        if self.categorical_cols:
            if self.cat_imputer is not None:
                cat_arr = self.cat_imputer.transform(X[list(self.categorical_cols)])
                cat_df = pd.DataFrame(cat_arr, columns=list(self.categorical_cols), index=X.index)
            else:
                cat_df = X[list(self.categorical_cols)].copy()

            for c in self.categorical_cols:
                # category dtype is important for LightGBM; CatBoost can take strings too.
                cat_df[c] = cat_df[c].astype("category")
            parts.append(cat_df)

        if not parts:
            return pd.DataFrame(index=X.index)

        return pd.concat(parts, axis=1)


class CatBoostSkWrapper(BaseEstimator, ClassifierMixin):
    """
    sklearn-compatible wrapper to pass cat_features at fit time.
    Expects X as a DataFrame.
    """

    def __init__(self, cat_cols: Tuple[str, ...], params: Optional[Dict[str, Any]] = None):
        self.cat_cols = tuple(cat_cols)
        self.params = dict(params or {})

    def fit(self, X, y):
        X_df = X.copy()
        self._model = CatBoostClassifier(**self.params)
        self._model.fit(X_df, y, cat_features=list(self.cat_cols))
        return self

    def predict(self, X):
        return self._model.predict(X.copy())

    def predict_proba(self, X):
        return self._model.predict_proba(X.copy())


class CatBoostRegWrapper(BaseEstimator):
    def __init__(self, cat_cols: Tuple[str, ...], params: Optional[Dict[str, Any]] = None):
        self.cat_cols = tuple(cat_cols)
        self.params = dict(params or {})

    def fit(self, X, y):
        X_df = X.copy()
        self._model = CatBoostRegressor(**self.params)
        self._model.fit(X_df, y, cat_features=list(self.cat_cols))
        return self

    def predict(self, X):
        return self._model.predict(X.copy())


class LGBMSkWrapper(BaseEstimator, ClassifierMixin):
    """
    Ensures categorical cols are 'category' dtype before fit/predict.
    """

    def __init__(self, cat_cols: Tuple[str, ...], params: Optional[Dict[str, Any]] = None):
        self.cat_cols = tuple(cat_cols)
        self.params = dict(params or {})

    def _cast(self, X: pd.DataFrame) -> pd.DataFrame:
        X_df = X.copy()
        for c in self.cat_cols:
            if c in X_df.columns:
                X_df[c] = X_df[c].astype("category")
        return X_df

    def fit(self, X, y):
        X_df = self._cast(X)
        self._model = LGBMClassifier(**self.params)
        self._model.fit(X_df, y, categorical_feature=list(self.cat_cols))
        return self

    def predict(self, X):
        return self._model.predict(self._cast(X))

    def predict_proba(self, X):
        return self._model.predict_proba(self._cast(X))


class LGBMRegWrapper(BaseEstimator):
    def __init__(self, cat_cols: Tuple[str, ...], params: Optional[Dict[str, Any]] = None):
        self.cat_cols = tuple(cat_cols)
        self.params = dict(params or {})

    def _cast(self, X: pd.DataFrame) -> pd.DataFrame:
        X_df = X.copy()
        for c in self.cat_cols:
            if c in X_df.columns:
                X_df[c] = X_df[c].astype("category")
        return X_df

    def fit(self, X, y):
        X_df = self._cast(X)
        self._model = LGBMRegressor(**self.params)
        self._model.fit(X_df, y, categorical_feature=list(self.cat_cols))
        return self

    def predict(self, X):
        return self._model.predict(self._cast(X))


## Config objects + Presets

In [63]:
TaskType = Literal["auto", "classification", "regression"]
EncoderChoice = Literal["onehot", "target", "native_xgb", "native_lgbm", "native_catboost"]
ModelChoice = Literal[
    "logreg", "rf", "xgb", "lgbm", "catboost",
    "gaussian_nb", "bernoulli_nb", "complement_nb", "categorical_nb",
    "linreg"
]
ImputerChoice = Literal["simple", "knn", "iterative", "none", "constant"]


@dataclass(frozen=True)
class RowFilter:
    col: str
    op: Literal[">=", ">", "<=", "<", "==", "!="]
    value: Any


@dataclass
class CVConfig:
    n_splits: int = 5
    n_repeats: int = 10
    shuffle: bool = True
    random_state: int = 42
    scoring: Optional[str] = None  # e.g., "roc_auc"


@dataclass
class FFSConfig:
    max_features: int = 30
    min_improvement: float = 0.0
    verbose: bool = True


@dataclass
class EncodingConfig:
    type: EncoderChoice = "onehot"
    params: Dict[str, Any] = field(default_factory=dict)


@dataclass
class ImputationConfig:
    numeric_type: ImputerChoice = "knn"
    numeric_params: Dict[str, Any] = field(default_factory=lambda: {"n_neighbors": 15})

    categorical_type: ImputerChoice = "simple"
    categorical_params: Dict[str, Any] = field(default_factory=lambda: {"strategy": "most_frequent"})


@dataclass
class ModelConfig:
    type: ModelChoice
    params: Dict[str, Any] = field(default_factory=dict)


@dataclass
class SelectorConfig:
    target_col: str = "ARIA.H_detected"
    task_type: TaskType = "auto"
    ignore_cols: List[str] = field(default_factory=list)

    min_non_missing_pct: Optional[float] = 0.15
    min_non_missing: Optional[int] = None
    row_filters: List[RowFilter] = field(default_factory=list)

    numeric_features: Optional[List[str]] = None
    categorical_features: Optional[List[str]] = None

    imputation: ImputationConfig = field(default_factory=ImputationConfig)
    encoding: EncodingConfig = field(default_factory=EncodingConfig)
    model: ModelConfig = field(default_factory=lambda: ModelConfig(type="xgb"))
    cv: CVConfig = field(default_factory=lambda: CVConfig(n_splits=5, scoring="average_precision"))
    ffs: FFSConfig = field(default_factory=FFSConfig)


# ---------------- Presets ----------------

MODEL_PRESETS: Dict[ModelChoice, ModelConfig] = {
    "logreg": ModelConfig(
        type="logreg",
        params=dict(
            l1_ratio=0,
            solver="lbfgs",
            C=0.3,
            max_iter=3000,
            tol=1e-3,
            class_weight="balanced",
            random_state=42,
        ),
    ),
    "rf": ModelConfig(
        type="rf",
        params=dict(
            n_estimators=600,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42,
            class_weight="balanced",
        ),
    ),
    "xgb": ModelConfig(
        type="xgb",
        params=dict(
            n_estimators=600,
            max_depth=6,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            n_jobs=-1,
            random_state=42,
            eval_metric="auc",
            use_label_encoder=False,
            scale_pos_weight=None,  # Will be calculated automatically
            early_stopping_rounds=None,  # Can be set via model_overrides
        ),
    ),
    "lgbm": ModelConfig(
        type="lgbm",
        params=dict(
            n_estimators=1200,
            learning_rate=0.02,
            num_leaves=31,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            n_jobs=-1,
            random_state=42,
            scale_pos_weight=None,  # Will be calculated automatically
            early_stopping_rounds=None,  # Can be set via model_overrides
        ),
    ),
    "catboost": ModelConfig(
        type="catboost",
        params=dict(
            iterations=2000,
            learning_rate=0.03,
            depth=6,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=False,
            # class_weights will be calculated automatically in _build_estimator
            early_stopping_rounds=None,  # Can be set via model_overrides
        ),
    ),
    "gaussian_nb": ModelConfig(type="gaussian_nb", params={}),
    "bernoulli_nb": ModelConfig(type="bernoulli_nb", params={}),
    "complement_nb": ModelConfig(type="complement_nb", params={}),
    "categorical_nb": ModelConfig(type="categorical_nb", params={}),
    "linreg": ModelConfig(type="linreg", params={}),
}

ENCODER_PRESETS: Dict[EncoderChoice, EncodingConfig] = {
    "onehot": EncodingConfig(type="onehot", params=dict(sparse_output=True)),
    "target": EncodingConfig(type="target", params=dict(
        handle_missing="value",
        handle_unknown="value",
        smoothing=0.2,
    )),
    "native_xgb": EncodingConfig(type="native_xgb", params={}),
    "native_lgbm": EncodingConfig(type="native_lgbm", params={}),
    "native_catboost": EncodingConfig(type="native_catboost", params={}),
}

IMPUTER_PRESETS: Dict[str, ImputationConfig] = {
    "knn15_freq": ImputationConfig(
        numeric_type="knn", numeric_params={"n_neighbors": 15},
        categorical_type="simple", categorical_params={"strategy": "most_frequent"},
    ),
    "median_freq": ImputationConfig(
        numeric_type="simple", numeric_params={"strategy": "median"},
        categorical_type="simple", categorical_params={"strategy": "most_frequent"},
    ),
    "iterative_freq": ImputationConfig(
        numeric_type="iterative", numeric_params={"random_state": 42, "max_iter": 10},
        categorical_type="simple", categorical_params={"strategy": "most_frequent"},
    ),
    "median_constant": ImputationConfig(
        numeric_type="simple", numeric_params={"strategy": "median"},
        categorical_type="constant", categorical_params={"fill_value": "_missing_"},
    ),
}


## Build config by choosing presets

In [64]:
def make_config(
        *,
        target_col: str,
        ignore_cols: List[str],
        task_type: TaskType = "auto",
        min_non_missing_pct: Optional[float] = 0.15,
        min_non_missing: Optional[int] = None,
        row_filters: Optional[List[RowFilter]] = None,

        model_choice: ModelChoice = "xgb",
        encoder_choice: EncoderChoice = "onehot",
        imputer_choice: str = "median_freq",

        cv: Optional[CVConfig] = None,
        ffs: Optional[FFSConfig] = None,

        # optional overrides (small, not full dicts)
        model_overrides: Optional[Dict[str, Any]] = None,
        encoder_overrides: Optional[Dict[str, Any]] = None,
) -> SelectorConfig:
    if model_choice not in MODEL_PRESETS:
        raise ValueError(f"Unknown model_choice: {model_choice}")
    if encoder_choice not in ENCODER_PRESETS:
        raise ValueError(f"Unknown encoder_choice: {encoder_choice}")
    if imputer_choice not in IMPUTER_PRESETS:
        raise ValueError(f"Unknown imputer_choice: {imputer_choice}")

    model_cfg = MODEL_PRESETS[model_choice]
    enc_cfg = ENCODER_PRESETS[encoder_choice]
    imp_cfg = IMPUTER_PRESETS[imputer_choice]

    # copy + override
    model_params = dict(model_cfg.params)
    if model_overrides:
        model_params.update(model_overrides)
    model_cfg_final = ModelConfig(type=model_cfg.type, params=model_params)

    enc_params = dict(enc_cfg.params)
    if encoder_overrides:
        enc_params.update(encoder_overrides)
    enc_cfg_final = EncodingConfig(type=enc_cfg.type, params=enc_params)

    cfg = SelectorConfig(
        target_col=target_col,
        task_type=task_type,
        ignore_cols=list(ignore_cols),

        min_non_missing_pct=min_non_missing_pct,
        min_non_missing=min_non_missing,
        row_filters=list(row_filters or []),

        imputation=imp_cfg,
        encoding=enc_cfg_final,
        model=model_cfg_final,

        cv=cv or CVConfig(n_splits=5, scoring="average_precision"),
        ffs=ffs or FFSConfig(max_features=25 , min_improvement=2e-8, verbose=True),
    )
    return cfg


def show_config(cfg: SelectorConfig) -> None:
    # pretty print for notebook
    cfg_dict = asdict(cfg)
    cfg_dict["row_filters"] = [asdict(rf) for rf in cfg.row_filters]
    display(pd.json_normalize(cfg_dict).T.rename(columns={0: "value"}))


## The selector engine

This is the FFS, a cleaned up and extended version for any model choices.

In [65]:
class FFSImputerSelector:
    def __init__(self, config: SelectorConfig):
        self.config = config
        self.selected_features_: List[str] = []
        self.history_: List[Dict[str, Any]] = []
        self.best_score_: Optional[float] = None
        self.task_type_: Optional[str] = None  # 'classification' or 'regression'

    def fit(self, df: pd.DataFrame) -> "FFSImputerSelector":
        cfg = self.config
        if cfg.target_col not in df.columns:
            raise ValueError(f"target_col='{cfg.target_col}' not found")

        data = df.copy()
        data = self._clean_dtype(data)
        data = self._apply_row_filters(data)

        X, y = self._prepare_X_y(data, cfg.target_col)
        X = self._clean_feature_matrix(X)

        self.task_type_ = self._infer_task_type(y, cfg.task_type)

        numeric_features, categorical_features = self._get_feature_types(X)
        if cfg.numeric_features is not None:
            numeric_features = [c for c in cfg.numeric_features if c in X.columns]
        if cfg.categorical_features is not None:
            categorical_features = [c for c in cfg.categorical_features if c in X.columns]

        all_features = numeric_features + categorical_features
        if not all_features:
            raise ValueError("No usable features after cleanup.")

        self._run_forward_selection(
            X=X, y=y,
            numeric_features=numeric_features,
            categorical_features=categorical_features,
            all_features=all_features,
        )
        return self

    # ---------------- Cleaning / typing ----------------

    def _clean_dtype(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()

        bool_cols = out.select_dtypes(include=["bool"]).columns.tolist()
        if bool_cols:
            out[bool_cols] = out[bool_cols].astype("object").astype("string")

        dt_cols = out.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
        for col in dt_cols:
            out[col] = out[col].view("int64").astype("float64")

        return out

    def _infer_task_type(self, y: pd.Series, task_type_cfg: TaskType) -> str:
        if task_type_cfg in ("classification", "regression"):
            return task_type_cfg
        unique_vals = pd.unique(y.dropna())
        if y.dtype.kind in "ifu" and len(unique_vals) <= 20:
            return "classification"
        return "regression"

    def _get_feature_types(self, X: pd.DataFrame) -> Tuple[List[str], List[str]]:
        num_cols = X.select_dtypes(include=["number"]).columns.tolist()
        cat_cols = [c for c in X.columns if c not in num_cols]
        return num_cols, cat_cols

    def _apply_row_filters(self, data: pd.DataFrame) -> pd.DataFrame:
        out = data
        for flt in self.config.row_filters:
            col, op, val = flt.col, flt.op, flt.value
            if col not in out.columns:
                continue
            if op == ">=":
                out = out[out[col] >= val]
            elif op == ">":
                out = out[out[col] > val]
            elif op == "<=":
                out = out[out[col] <= val]
            elif op == "<":
                out = out[out[col] < val]
            elif op == "==":
                out = out[out[col] == val]
            elif op == "!=":
                out = out[out[col] != val]
            else:
                raise ValueError(f"Unknown op: {op}")
        return out

    def _prepare_X_y(self, data: pd.DataFrame, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
        data = data[~data[target_col].isna()].reset_index(drop=True)
        y = data[target_col]
        X = data.drop(columns=[target_col])
        return X, y

    def _clean_feature_matrix(self, X: pd.DataFrame) -> pd.DataFrame:
        cfg = self.config
        out = X.copy()

        # drop ignore columns
        ignore = [c for c in cfg.ignore_cols if c in out.columns]
        if ignore:
            out = out.drop(columns=ignore)

        non_missing = out.notna().sum()
        n_rows = len(out)

        all_nan_cols = non_missing[non_missing == 0].index.tolist()

        thresholds = []
        if cfg.min_non_missing is not None:
            thresholds.append(int(cfg.min_non_missing))
        if cfg.min_non_missing_pct is not None:
            frac = cfg.min_non_missing_pct if cfg.min_non_missing_pct <= 1 else cfg.min_non_missing_pct / 100.0
            thresholds.append(int(np.ceil(frac * n_rows)))

        low_support_cols = []
        if thresholds:
            thr = max(thresholds)
            low_support_cols = non_missing[non_missing < thr].index.tolist()

        cols_to_drop = sorted(set(all_nan_cols + low_support_cols))
        if cols_to_drop and cfg.ffs.verbose:
            tqdm.write(f"[FFS] Dropped {len(cols_to_drop)} low-support/all-NaN columns.")
        if cols_to_drop:
            out = out.drop(columns=cols_to_drop)

        return out

    # ---------------- Build preprocessors / models ----------------

    def _make_numeric_imputer(self) -> BaseEstimator:
        imp = self.config.imputation
        if imp.numeric_type == "simple":
            params = dict(imp.numeric_params)
            params.setdefault("strategy", "median")
            return SimpleImputer(**params)
        if imp.numeric_type == "knn":
            return KNNImputer(**imp.numeric_params)
        if imp.numeric_type == "iterative":
            return IterativeImputer(**imp.numeric_params)
        raise ValueError(f"Unknown numeric imputer: {imp.numeric_type}")

    def _make_categorical_imputer(self) -> Optional[BaseEstimator]:
        imp = self.config.imputation
        t = imp.categorical_type
        params = dict(imp.categorical_params)

        if t == "simple":
            params.setdefault("strategy", "most_frequent")
            return SimpleImputer(**params)
        if t == "constant":
            fill_value = params.pop("fill_value", "_missing_")
            return SimpleImputer(strategy="constant", fill_value=fill_value, **params)
        if t == "none":
            return None
        raise ValueError(f"Unknown categorical imputer: {t}")

    def _make_encoder(self) -> BaseEstimator:
        enc = self.config.encoding
        if enc.type == "onehot":
            p = dict(enc.params)
            # force safe default
            p.pop("handle_unknown", "None")
            return OneHotEncoder(handle_unknown="ignore", **p)
        if enc.type == "target":
            return TargetEncoder(**enc.params)
        raise ValueError(f"Encoder '{enc.type}' is native-only or unsupported here.")

    def _build_preprocessor(
            self,
            numeric_features: List[str],
            categorical_features: List[str],
            used_features: List[str],
    ) -> BaseEstimator:
        num_used = [c for c in numeric_features if c in used_features]
        cat_used = [c for c in categorical_features if c in used_features]

        num_imputer = self._make_numeric_imputer()
        cat_imputer = self._make_categorical_imputer()

        enc_type = self.config.encoding.type

        # ---------- native modes (keep DataFrame) ----------
        if enc_type in ("native_xgb", "native_lgbm", "native_catboost"):
            return NativeDataFramePreprocessor(
                numeric_cols=tuple(num_used),
                categorical_cols=tuple(cat_used),
                num_imputer=num_imputer,
                cat_imputer=cat_imputer,
            )

        # ---------- encoded mode ----------
        transformers = []

        if num_used:
            num_pipe = Pipeline(steps=[
                ("imputer", num_imputer),
                ("scaler", StandardScaler(with_mean=False)),
            ])
            transformers.append(("num", num_pipe, num_used))

        if cat_used:
            cat_steps = []
            if cat_imputer is not None:
                cat_steps.append(("imputer", cat_imputer))
            cat_steps.append(("encoder", self._make_encoder()))
            cat_pipe = Pipeline(steps=cat_steps)
            transformers.append(("cat", cat_pipe, cat_used))

        if not transformers:
            raise ValueError("No features left for preprocessing.")

        ct = ColumnTransformer(transformers=transformers)

        # If GaussianNB requires dense matrix, convert.
        if self.config.model.type == "gaussian_nb":
            def to_dense(x):
                return x.toarray() if hasattr(x, "toarray") else np.asarray(x)

            ct = Pipeline(steps=[("ct", ct), ("to_dense", FunctionTransformer(to_dense, accept_sparse=True))])

        return ct

    def _calculate_class_weights(self, y: pd.Series) -> Optional[Dict[int, float]]:
        """Calculate class weights dict for CatBoost from class distribution."""
        if y is None:
            return None
        y_array = np.asarray(y)
        unique_classes = np.unique(y_array)
        if len(unique_classes) != 2:
            return None
        
        n_neg = (y_array == 0).sum()
        n_pos = (y_array == 1).sum()
        if n_pos > 0 and n_neg > 0:
            # Calculate weight ratio: negative class gets weight 1.0, positive gets n_neg/n_pos
            weight_ratio = n_neg / n_pos
            return {0: 1.0, 1: weight_ratio}
        return None

    def _build_estimator(self, categorical_cols_used: List[str], y: Optional[pd.Series] = None) -> BaseEstimator:
        cfg = self.config
        params = dict(cfg.model.params)

        # classification vs regression - determine with multiple fallbacks
        if self.task_type_ is not None:
            is_clf = (self.task_type_ == "classification")
        elif cfg.task_type in ("classification", "regression"):
            is_clf = (cfg.task_type == "classification")
        elif y is not None:
            # Infer from y if task_type_ not set yet
            is_clf = self._infer_task_type(y, cfg.task_type) == "classification"
        else:
            # Default fallback: if using classification scoring, assume classification
            if cfg.cv.scoring in ("average_precision", "roc_auc", "f1", "precision", "recall"):
                is_clf = True
            else:
                is_clf = False
        
        # Additional safeguard: if using classification scoring, force classification
        if cfg.cv.scoring in ("average_precision", "roc_auc", "f1", "precision", "recall"):
            is_clf = True
        
        # Calculate class weights for imbalanced datasets (classification only)
        if is_clf and y is not None:
            y_array = np.asarray(y)
            if len(np.unique(y_array)) == 2:  # Binary classification
                n_neg = (y_array == 0).sum()
                n_pos = (y_array == 1).sum()
                if n_pos > 0 and n_neg > 0:
                    scale_pos_weight = n_neg / n_pos
                else:
                    scale_pos_weight = 1.0
            else:
                scale_pos_weight = 1.0
        else:
            scale_pos_weight = None
        
        # Calculate class_weights for CatBoost (try even if is_clf might be wrong, as a check)
        class_weights = self._calculate_class_weights(y) if (is_clf or cfg.cv.scoring in ("average_precision", "roc_auc", "f1", "precision", "recall")) and y is not None else None
        
        # If class_weights were calculated, this must be classification
        if class_weights is not None:
            is_clf = True

        # ---------- Logistic / Linear ----------
        if cfg.model.type == "logreg":
            if not is_clf:
                raise ValueError("logreg is classification-only.")
            params.setdefault("max_iter", 2000)
            return LogisticRegression(**params)

        if cfg.model.type == "linreg":
            if is_clf:
                raise ValueError("linreg is regression-only.")
            return LinearRegression(**params)

        # ---------- RandomForest ----------
        if cfg.model.type == "rf":
            return RandomForestClassifier(**params) if is_clf else RandomForestRegressor(**params)

        # ---------- XGBoost ----------
        if cfg.model.type == "xgb":
            # Remove early_stopping_rounds from params if present (handled separately in CV)
            early_stopping = params.pop("early_stopping_rounds", None)
            
            if is_clf:
                params.setdefault("eval_metric", "auc")
                params.setdefault("random_state", 42)
                params.setdefault("use_label_encoder", False)
                # Set scale_pos_weight if not explicitly provided and we have class distribution
                if scale_pos_weight is not None and params.get("scale_pos_weight") is None:
                    params["scale_pos_weight"] = scale_pos_weight
                if cfg.encoding.type == "native_xgb":
                    # NOTE: XGB native categorical remains fragile in CV if unseen categories appear.
                    params.setdefault("enable_categorical", True)
                    params.setdefault("tree_method", "hist")
                return XGBClassifier(**params)
            else:
                params.setdefault("eval_metric", "rmse")
                params.setdefault("random_state", 42)
                if cfg.encoding.type == "native_xgb":
                    params.setdefault("enable_categorical", True)
                    params.setdefault("tree_method", "hist")
                return XGBRegressor(**params)

        # ---------- LightGBM ----------
        if cfg.model.type == "lgbm":
            # Remove early_stopping_rounds from params if present (handled separately in CV)
            early_stopping = params.pop("early_stopping_rounds", None)
            
            if is_clf:
                # Set scale_pos_weight if not explicitly provided and we have class distribution
                if scale_pos_weight is not None and params.get("scale_pos_weight") is None:
                    params["scale_pos_weight"] = scale_pos_weight
            if cfg.encoding.type == "native_lgbm":
                # wrapper expects DataFrame
                return LGBMSkWrapper(tuple(categorical_cols_used), params) if is_clf else LGBMRegWrapper(
                    tuple(categorical_cols_used), params)
            return LGBMClassifier(**params) if is_clf else LGBMRegressor(**params)

        # ---------- CatBoost ----------
        if cfg.model.type == "catboost":
            # Remove early_stopping_rounds from params if present (handled separately)
            early_stopping = params.pop("early_stopping_rounds", None)
            
            # Force classification if class_weights are being set or classification scoring is used
            if class_weights is not None or cfg.cv.scoring in ("average_precision", "roc_auc", "f1", "precision", "recall"):
                is_clf = True
            
            # Set class_weights for classification if not explicitly provided
            if is_clf and class_weights is not None and params.get("class_weights") is None:
                params["class_weights"] = class_weights
            
            if cfg.encoding.type == "native_catboost":
                return CatBoostSkWrapper(tuple(categorical_cols_used), params) if is_clf else CatBoostRegWrapper(
                    tuple(categorical_cols_used), params)
            return CatBoostClassifier(**params) if is_clf else CatBoostRegressor(**params)

        # ---------- Bayesian ----------
        if not is_clf:
            raise ValueError(f"Bayesian models are classification-only; got {cfg.model.type}")

        if cfg.model.type == "gaussian_nb":
            return GaussianNB(**params)
        if cfg.model.type == "bernoulli_nb":
            return BernoulliNB(**params)
        if cfg.model.type == "complement_nb":
            return ComplementNB(**params)
        if cfg.model.type == "categorical_nb":
            return CategoricalNB(**params)

        raise ValueError(f"Unknown model type: {cfg.model.type}")

    def _build_cv(self):
        cv_cfg = self.config.cv
        if self.task_type_ == "classification":
            return RepeatedStratifiedKFold(n_splits=cv_cfg.n_splits, n_repeats=cv_cfg.n_repeats, random_state=cv_cfg.random_state)
        return KFold(n_splits=cv_cfg.n_splits, shuffle=cv_cfg.shuffle, random_state=cv_cfg.random_state)

    # ---------------- FFS core ----------------

    def _run_forward_selection(
            self,
            X: pd.DataFrame,
            y: pd.Series,
            numeric_features: List[str],
            categorical_features: List[str],
            all_features: List[str],
    ) -> None:
        cfg = self.config
        remaining_features = all_features.copy()
        current_best = -np.inf

        outer_iter = tqdm(range(cfg.ffs.max_features), desc="FFS steps", unit="step",
                          leave=True) if cfg.ffs.verbose else range(cfg.ffs.max_features)

        for step_idx in outer_iter:
            if not remaining_features:
                break

            best_feature = None
            best_score_this_round = current_best

            for feat in remaining_features:
                candidate_features = self.selected_features_ + [feat]
                score = self._evaluate_feature_subset(
                    X=X, y=y,
                    candidate_features=candidate_features,
                    numeric_features=numeric_features,
                    categorical_features=categorical_features,
                )

                self.history_.append({
                    "step": len(self.selected_features_) + 1,
                    "candidate": feat,
                    "n_features": len(candidate_features),
                    "score": float(score),
                })

                if score > best_score_this_round:
                    best_score_this_round = score
                    best_feature = feat

            improvement = best_score_this_round - current_best

            if best_feature is not None and improvement >= cfg.ffs.min_improvement:
                self.selected_features_.append(best_feature)
                remaining_features.remove(best_feature)
                current_best = best_score_this_round
                self.best_score_ = current_best

                if cfg.ffs.verbose:
                    outer_iter.set_postfix(
                        {"last_added": best_feature, "best": f"{current_best:.5f}", "imp": f"{improvement:.6g}"})
                    tqdm.write(
                        f"[FFS] Step {step_idx + 1}: +{best_feature} | score={current_best:.5f} | Δ={improvement:.6g}")
            else:
                if cfg.ffs.verbose:
                    tqdm.write(
                        f"[FFS] Stop at step {step_idx + 1}: Δ={improvement:.6g} < min_improvement={cfg.ffs.min_improvement:.6g}")
                break

    def _evaluate_feature_subset(
            self,
            X: pd.DataFrame,
            y: pd.Series,
            candidate_features: List[str],
            numeric_features: List[str],
            categorical_features: List[str],
    ) -> float:
        preprocessor = self._build_preprocessor(
            numeric_features=numeric_features,
            categorical_features=categorical_features,
            used_features=candidate_features,
        )

        cat_used = [c for c in categorical_features if c in candidate_features]
        estimator = self._build_estimator(categorical_cols_used=cat_used, y=y)

        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ])

        cv = self._build_cv()
        scoring = self.config.cv.scoring
        if scoring is None:
            scoring = "average_precision" if self.task_type_ == "classification" else "neg_root_mean_squared_error"

        scores = cross_val_score(pipe, X[candidate_features], y, cv=cv, scoring=scoring)
        return float(np.mean(scores))


## Threshold Optimization for Imbalanced Classification


In [66]:
class ThresholdOptimizer:
    """
    Finds optimal classification threshold for imbalanced binary classification.
    Optimizes based on PR-AUC, F1-score, or custom metric.
    """
    
    def __init__(self, metric: str = "f1"):
        """
        Args:
            metric: Metric to optimize ('f1', 'pr_auc', 'precision', 'recall', or custom callable)
        """
        self.metric = metric
        self.optimal_threshold_: Optional[float] = None
        self.optimal_score_: Optional[float] = None
    
    def find_optimal_threshold(
        self, 
        y_true: np.ndarray, 
        y_proba: np.ndarray, 
        thresholds: Optional[np.ndarray] = None
    ) -> float:
        """
        Find optimal threshold that maximizes the specified metric.
        
        Args:
            y_true: True binary labels
            y_proba: Predicted probabilities for positive class
            thresholds: Optional array of thresholds to test. If None, uses 0.1 to 0.9 in 0.01 steps.
        
        Returns:
            Optimal threshold value
        """
        if thresholds is None:
            thresholds = np.arange(0.1, 0.9, 0.01)
        
        best_threshold = 0.5
        best_score = -np.inf
        
        for threshold in thresholds:
            y_pred = (y_proba >= threshold).astype(int)
            
            if self.metric == "f1":
                score = f1_score(y_true, y_pred, zero_division=0)
            elif self.metric == "pr_auc":
                # For PR-AUC, we need to compute precision-recall curve
                precision, recall, _ = precision_recall_curve(y_true, y_proba)
                score = np.trapz(precision, recall)  # Approximate PR-AUC
            elif self.metric == "precision":
                score = precision_score(y_true, y_pred, zero_division=0)
            elif self.metric == "recall":
                score = recall_score(y_true, y_pred, zero_division=0)
            elif callable(self.metric):
                score = self.metric(y_true, y_pred)
            else:
                raise ValueError(f"Unknown metric: {self.metric}")
            
            if score > best_score:
                best_score = score
                best_threshold = threshold
        
        self.optimal_threshold_ = best_threshold
        self.optimal_score_ = best_score
        return best_threshold
    
    def predict(self, y_proba: np.ndarray, threshold: Optional[float] = None) -> np.ndarray:
        """
        Apply threshold to probabilities to get binary predictions.
        
        Args:
            y_proba: Predicted probabilities
            threshold: Threshold to use. If None, uses optimal_threshold_
        
        Returns:
            Binary predictions
        """
        if threshold is None:
            if self.optimal_threshold_ is None:
                raise ValueError("No optimal threshold found. Call find_optimal_threshold first.")
            threshold = self.optimal_threshold_
        return (y_proba >= threshold).astype(int)


## Ensemble Methods for FFS


In [67]:
class EnsembleFFSSelector:
    """
    Combines multiple FFSImputerSelector results using voting or stacking.
    """
    
    def __init__(
        self, 
        selectors: List[FFSImputerSelector],
        method: Literal["voting", "stacking"] = "voting",
        weights: Optional[List[float]] = None
    ):
        """
        Args:
            selectors: List of fitted FFSImputerSelector instances
            method: 'voting' for weighted voting or 'stacking' for meta-learner
            weights: Optional weights for voting. If None, uses CV scores as weights.
        """
        self.selectors = selectors
        self.method = method
        self.weights = weights
        self.meta_learner_ = None
        self._validate_selectors()
    
    def _validate_selectors(self):
        """Ensure all selectors are fitted and compatible."""
        if not self.selectors:
            raise ValueError("At least one selector required")
        
        # Check all are fitted
        for sel in self.selectors:
            if not hasattr(sel, 'selected_features_') or not sel.selected_features_:
                raise ValueError("All selectors must be fitted before creating ensemble")
        
        # Check same task type
        task_types = [sel.task_type_ for sel in self.selectors]
        if len(set(task_types)) > 1:
            raise ValueError("All selectors must have the same task_type")
        
        self.task_type_ = task_types[0]
        if self.task_type_ != "classification":
            raise ValueError("Ensemble currently only supports classification")
    
    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """
        Get ensemble predictions.
        
        Args:
            X: Feature matrix
        
        Returns:
            Predicted probabilities
        """
        if self.method == "voting":
            return self._weighted_voting(X)
        elif self.method == "stacking":
            return self._stacking_predict(X)
        else:
            raise ValueError(f"Unknown method: {self.method}")
    
    def _weighted_voting(self, X: pd.DataFrame) -> np.ndarray:
        """Weighted average of predictions from all selectors."""
        all_probas = []
        weights = self.weights
        
        # If no weights provided, use CV scores
        if weights is None:
            weights = [sel.best_score_ or 1.0 for sel in self.selectors]
            # Normalize weights
            total = sum(weights)
            weights = [w / total for w in weights]
        
        for sel, weight in zip(self.selectors, weights):
            # Build pipeline for each selector
            preprocessor = sel._build_preprocessor(
                numeric_features=sel._get_feature_types(X)[0],
                categorical_features=sel._get_feature_types(X)[1],
                used_features=sel.selected_features_,
            )
            
            cat_used = [c for c in sel._get_feature_types(X)[1] if c in sel.selected_features_]
            estimator = sel._build_estimator(categorical_cols_used=cat_used, y=None)
            
            pipe = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("model", estimator),
            ])
            
            # Fit on full data (assuming selectors were already fitted)
            # For prediction, we need to refit or store fitted pipelines
            # This is a simplified version - in practice, you'd store fitted pipelines
            proba = pipe.fit(X[sel.selected_features_], np.zeros(len(X))).predict_proba(X[sel.selected_features_])
            all_probas.append(proba * weight)
        
        return np.sum(all_probas, axis=0)
    
    def _stacking_predict(self, X: pd.DataFrame) -> np.ndarray:
        """Stacking with meta-learner predictions."""
        if self.meta_learner_ is None:
            raise ValueError("Meta-learner not fitted. Call fit_meta_learner first.")
        
        # Get base predictions
        base_preds = []
        for sel in self.selectors:
            preprocessor = sel._build_preprocessor(
                numeric_features=sel._get_feature_types(X)[0],
                categorical_features=sel._get_feature_types(X)[1],
                used_features=sel.selected_features_,
            )
            cat_used = [c for c in sel._get_feature_types(X)[1] if c in sel.selected_features_]
            estimator = sel._build_estimator(categorical_cols_used=cat_used, y=None)
            pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
            proba = pipe.fit(X[sel.selected_features_], np.zeros(len(X))).predict_proba(X[sel.selected_features_])
            base_preds.append(proba[:, 1])  # Positive class probability
        
        # Stack predictions
        X_meta = np.column_stack(base_preds)
        return self.meta_learner_.predict_proba(X_meta)
    
    def fit_meta_learner(self, X: pd.DataFrame, y: pd.Series, meta_learner=None):
        """
        Fit meta-learner for stacking.
        
        Args:
            X: Training features
            y: Training labels
            meta_learner: Meta-learner model. If None, uses LogisticRegression.
        """
        if meta_learner is None:
            meta_learner = LogisticRegression(random_state=42, max_iter=1000)
        
        # Get base predictions using CV
        base_preds = []
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        
        for sel in self.selectors:
            preprocessor = sel._build_preprocessor(
                numeric_features=sel._get_feature_types(X)[0],
                categorical_features=sel._get_feature_types(X)[1],
                used_features=sel.selected_features_,
            )
            cat_used = [c for c in sel._get_feature_types(X)[1] if c in sel.selected_features_]
            estimator = sel._build_estimator(categorical_cols_used=cat_used, y=y)
            pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
            
            # Get out-of-fold predictions
            oof_preds = np.zeros(len(X))
            for train_idx, val_idx in cv.split(X[sel.selected_features_], y):
                pipe.fit(X[sel.selected_features_].iloc[train_idx], y.iloc[train_idx])
                oof_preds[val_idx] = pipe.predict_proba(X[sel.selected_features_].iloc[val_idx])[:, 1]
            
            base_preds.append(oof_preds)
        
        # Fit meta-learner
        X_meta = np.column_stack(base_preds)
        self.meta_learner_ = meta_learner.fit(X_meta, y)


## Enhanced Evaluation Metrics


In [68]:
def compute_comprehensive_metrics(
    y_true: np.ndarray, 
    y_pred: np.ndarray, 
    y_proba: Optional[np.ndarray] = None
) -> Dict[str, float]:
    """
    Compute comprehensive classification metrics.
    
    Args:
        y_true: True binary labels
        y_pred: Predicted binary labels
        y_proba: Predicted probabilities (optional, for AUC metrics)
    
    Returns:
        Dictionary of metric names and values
    """
    metrics = {}
    
    # Basic metrics
    metrics["f1"] = f1_score(y_true, y_pred, zero_division=0)
    metrics["precision"] = precision_score(y_true, y_pred, zero_division=0)
    metrics["recall"] = recall_score(y_true, y_pred, zero_division=0)
    
    # AUC metrics (require probabilities)
    if y_proba is not None:
        metrics["roc_auc"] = roc_auc_score(y_true, y_proba)
        metrics["pr_auc"] = average_precision_score(y_true, y_proba)
    
    return metrics


def evaluate_model_comprehensive(
    estimator: BaseEstimator,
    X: pd.DataFrame,
    y: pd.Series,
    cv,
    scoring: Optional[str] = None
) -> Dict[str, Any]:
    """
    Evaluate model with comprehensive metrics using cross-validation.
    
    Args:
        estimator: Fitted or unfitted estimator
        X: Feature matrix
        y: Target labels
        cv: Cross-validation splitter
        scoring: Primary scoring metric (default: average_precision for classification)
    
    Returns:
        Dictionary with mean scores and detailed metrics
    """
    if scoring is None:
        # Infer task type
        unique_vals = pd.unique(y.dropna())
        if len(unique_vals) <= 20:
            scoring = "average_precision"
        else:
            scoring = "neg_root_mean_squared_error"
    
    # Primary score
    scores = cross_val_score(estimator, X, y, cv=cv, scoring=scoring)
    primary_score = float(np.mean(scores))
    
    # Comprehensive metrics (for classification only)
    detailed_metrics = {}
    if scoring in ["average_precision", "roc_auc", "f1", "precision", "recall"]:
        # Get predictions for detailed metrics
        y_preds = []
        y_probas = []
        y_trues = []
        
        for train_idx, val_idx in cv.split(X, y):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            estimator.fit(X_train, y_train)
            y_pred = estimator.predict(X_val)
            y_proba = estimator.predict_proba(X_val)[:, 1] if hasattr(estimator, "predict_proba") else None
            
            y_preds.extend(y_pred)
            if y_proba is not None:
                y_probas.extend(y_proba)
            y_trues.extend(y_val.values)
        
        if y_probas:
            detailed_metrics = compute_comprehensive_metrics(
                np.array(y_trues),
                np.array(y_preds),
                np.array(y_probas)
            )
        else:
            detailed_metrics = compute_comprehensive_metrics(
                np.array(y_trues),
                np.array(y_preds),
                None
            )
    
    return {
        "primary_score": primary_score,
        "primary_metric": scoring,
        "detailed_metrics": detailed_metrics,
        "cv_scores": scores.tolist(),
        "cv_mean": primary_score,
        "cv_std": float(np.std(scores))
    }


## Keep in touch with data

In [69]:
input_path = "../data/raw/VLST.csv"
df = pd.read_csv(input_path)

df.shape, df.columns[:10]


((5185, 85),
 Index(['NO.', 'Stent thrombosis', 'Name', 'Age', 'Men',
        'Time since stent implantation', 'Diabetes', 'Hypertension', 'HL',
        'Current smoker'],
       dtype='object'))

## Choose config by options

##### Here you only select:

1. model_choice

2. encoder_choice (onehot/target/native)

3. imputer_choice

#### and small overrides if needed.

In [74]:
IGNORE_COLS = [
    "NO.",
    "Name",
]

# ROW_FILTERS = [
#     RowFilter(col="X..of.Completed.Lecanemab.Infusions", op=">=", value=6)
# ]

cfg = make_config(
    target_col="Stent thrombosis",
    ignore_cols=IGNORE_COLS,
    task_type="classification",

    min_non_missing_pct=0.1,
    # row_filters=ROW_FILTERS,

    # Choose here:
    model_choice="rf",  # "xgb", "catboost", "logreg", "gaussian_nb", "rf", ...
    encoder_choice="onehot",  # "onehot" or "target" (or "native_lgbm"/"native_catboost"/"native_xgb")
    imputer_choice="median_freq",  # "knn15_freq", "iterative_freq", ...

    # Optional overrides (small)
    # model_overrides={
    #     "n_estimators": 800,
    #     "learning_rate": 0.03,
    #     "min_child_samples": 5,
    #     "min_split_gain": 0.0,
    #     "class_weight": "balanced",
    # },
    # encoder_overrides={"smoothing": 0.25},
    ffs=FFSConfig(max_features=30, min_improvement=2e-5, verbose=True),
    cv=CVConfig(n_splits=5, n_repeats=10, shuffle=True, random_state=42, scoring="average_precision"),
)

show_config(cfg)


,value
target_col,Stent thrombosis
task_type,classification
ignore_cols,"[NO., Name]"
min_non_missing_pct,0.1
min_non_missing,None
row_filters,[]
numeric_features,None
categorical_features,None
imputation.numeric_type,simple
imputation.numeric_params.strategy,median


## Run FFS

In [75]:
selector = FFSImputerSelector(cfg)
selector.fit(df)

print("Selected features:", selector.selected_features_)
print("Best CV score:", selector.best_score_)

history_df = pd.DataFrame(selector.history_)
history_df.head()

FFS steps:   3%|▎         | 1/30 [35:39<17:13:54, 2139.12s/step, last_added=Time since stent implantation, best=0.76581, imp=inf]

[FFS] Step 1: +Time since stent implantation | score=0.76581 | Δ=inf


FFS steps:   7%|▋         | 2/30 [2:44:09<42:07:33, 5416.19s/step, last_added=CaI, best=0.83787, imp=0.0720552]                  

[FFS] Step 2: +CaI | score=0.83787 | Δ=0.0720552


FFS steps:  10%|█         | 3/30 [3:36:44<32:52:39, 4383.69s/step, last_added=Stent type-SES, best=0.89250, imp=0.0546292]

...

FFS steps:  67%|██████▋   | 20/30 [33:26:57<30:47:51, 11087.17s/step, last_added=CKD5, best=0.96837, imp=0.000176925]   

[FFS] Step 20: +CKD5 | score=0.96837 | Δ=0.000176925


FFS steps:  67%|██████▋   | 20/30 [40:45:12<20:22:36, 7335.64s/step, last_added=CKD5, best=0.96837, imp=0.000176925] 

[FFS] Stop at step 21: Δ=0 < min_improvement=2e-05
Selected features: ['Time since stent implantation', 'CaI', 'Stent type-SES', 'WBC', 'No postdilation', 'eGFR', 'Pre-TIMI flow-3', 'Visual thrombus', 'Cre', 'LDL', 'STEMI', 'LV', 'LVEF', 'HGB', 'Thrombus aspiration', 'Men', 'No.of stents per lesion', 'UA', 'Slow flow', 'CKD5']
Best CV score: 0.9683661432739311


,step,candidate,n_features,score
0,1,Age,1,0.023866
1,1,Men,1,0.018993
2,1,Time since stent implantation,1,0.765811
3,1,Diabetes,1,0.022080
4,1,Hypertension,1,0.017515


## Compare multiple models

This lets you select multiple models and run FFS for each, then compare best scores and selected features.

In [41]:
def run_ffs_for_model_choices(
        df: pd.DataFrame,
        base_cfg: SelectorConfig,
        model_choices: List[ModelChoice],
        encoder_choice: Optional[EncoderChoice] = None,
        imputer_choice: Optional[str] = None,
) -> pd.DataFrame:
    results = []
    for mc in model_choices:
        cfg = make_config(
            target_col=base_cfg.target_col,
            ignore_cols=base_cfg.ignore_cols,
            task_type=base_cfg.task_type,
            min_non_missing_pct=base_cfg.min_non_missing_pct,
            min_non_missing=base_cfg.min_non_missing,
            row_filters=base_cfg.row_filters,
            model_choice=mc,
            encoder_choice=encoder_choice or base_cfg.encoding.type,
            imputer_choice=imputer_choice or "median_freq",
            cv=base_cfg.cv,
            ffs=base_cfg.ffs,
        )
        sel = FFSImputerSelector(cfg).fit(df)
        results.append({
            "model": mc,
            "encoder": cfg.encoding.type,
            "best_score": sel.best_score_,
            "n_selected": len(sel.selected_features_),
            "selected_features": sel.selected_features_,
        })
    return pd.DataFrame(results).sort_values("best_score", ascending=False).reset_index(drop=True)


comparison = run_ffs_for_model_choices(
    df=df,
    base_cfg=cfg,
    model_choices=["logreg", "xgb", "catboost", "gaussian_nb", "rf"],
    encoder_choice="onehot",  # keep encoding fixed for fairness
    imputer_choice="median_freq",
)

comparison

FFS steps:   3%|▎         | 1/30 [00:17<08:15, 17.07s/step, last_added=Time since stent implantation, best=0.77685, imp=inf]

[FFS] Step 1: +Time since stent implantation | score=0.77685 | Δ=inf


FFS steps:   7%|▋         | 2/30 [00:39<09:32, 20.45s/step, last_added=No postdilation, best=0.78189, imp=0.0050361]        

[FFS] Step 2: +No postdilation | score=0.78189 | Δ=0.0050361


FFS steps:  10%|█         | 3/30 [01:03<09:46, 21.71s/step, last_added=No.of stents per lesion, best=0.78807, imp=0.00618664]

...

[FFS] Step 11: +LVEF | score=0.94726 | Δ=0.00825784


FFS steps:  37%|███▋      | 11/30 [21:00:55<36:17:57, 6877.75s/step, last_added=LVEF, best=0.94726, imp=0.00825784]


[FFS] Stop at step 12: Δ=0 < min_improvement=2e-05


FFS steps:   0%|          | 0/30 [00:00<?, ?step/s]/Users/am.daraei/Projects/personal/vlst/.venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/am.daraei/Projects/personal/vlst/.venv/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "/Users/am.daraei/Projects/personal/vlst/.venv/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "/Users/am.daraei/Projects/personal/vlst/.venv/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ~~~~~~~~~~~~~

KeyboardInterrupt: 

In [29]:
for v in comparison.selected_features.values:
    print(v)

['days_since_1st_lecanemab_infusion', 'X..of.Completed.Lecanemab.Infusions', 'mri_brain_images_note.microhemorrhages_count', 'quick_dementia_rating_system_QDRS.function_at_home_and_hobby_activities', 'laboratory_testing_value.GLOB', 'functional_activities_questionnaire.remembering_appointments_or_medications', 'functional_activities_questionnaire.playing_game_or_hobby', 'laboratory_testing_value.WBC', 'examination_of_vital_signs.temperature']
['X..of.Completed.Lecanemab.Infusions', 'Abeta42_Roche_equivalent', 'alcohol_use.drinking_status_binary', 'diagnostic_clinically_syndrome_logopenic_variant_ppa', 'diagnostic_clinically_initialdomain_executive', 'diagnostic_clinically_syndrome_behavioral', 'focused_review_of_symptoms.motor_weakness', 'mri_brain_images_note.macrohemorrhages_count']
['days_since_1st_lecanemab_infusion', 'mri_brain_images_note.microhemorrhages_count', 'X..of.Completed.Lecanemab.Infusions', 'focused_review_of_symptoms.gait_disorder', 'CDR.CDR_global', 'diagnostic_clini

## Usage Examples for New Features

### Using Threshold Optimization

After training a model, you can find the optimal threshold:

```python
# After fitting a selector
selector = FFSImputerSelector(cfg).fit(df)

# Get predictions on validation set
# (you'd need to refit the model on full data or use CV predictions)
# For demonstration:
# y_proba = model.predict_proba(X_val)[:, 1]

# Find optimal threshold
threshold_opt = ThresholdOptimizer(metric="f1")
optimal_threshold = threshold_opt.find_optimal_threshold(y_val, y_proba)
print(f"Optimal threshold: {optimal_threshold:.3f}")

# Apply threshold
y_pred_optimized = threshold_opt.predict(y_proba)
```

### Using Ensemble Methods

```python
# Train multiple selectors with different models
selectors = []
for model in ["xgb", "lgbm", "catboost"]:
    cfg = make_config(..., model_choice=model)
    sel = FFSImputerSelector(cfg).fit(df)
    selectors.append(sel)

# Create ensemble with weighted voting
ensemble = EnsembleFFSSelector(selectors, method="voting")

# Or use stacking
ensemble = EnsembleFFSSelector(selectors, method="stacking")
ensemble.fit_meta_learner(X_train, y_train)

# Get ensemble predictions
y_proba_ensemble = ensemble.predict_proba(X_test)
```

### Enhanced Evaluation Metrics

```python
# Use comprehensive evaluation
results = evaluate_model_comprehensive(
    estimator=pipe,
    X=X[selected_features],
    y=y,
    cv=cv,
    scoring="average_precision"
)

print(f"PR-AUC: {results['primary_score']:.4f}")
print(f"ROC-AUC: {results['detailed_metrics'].get('roc_auc', 'N/A')}")
print(f"F1: {results['detailed_metrics'].get('f1', 'N/A')}")
```

### Notes on Early Stopping

Early stopping parameters (`early_stopping_rounds`) can be set via `model_overrides`:

```python
cfg = make_config(
    ...,
    model_choice="xgb",
    model_overrides={"early_stopping_rounds": 50}
)
```

**Note**: Early stopping in cross-validation context requires special handling. The current implementation extracts the parameter but doesn't fully utilize it in CV. For production use, consider:
1. Using a separate validation set
2. Implementing custom CV with eval_set support
3. Using the model's built-in early stopping after feature selection
